# Trained-pilot skill forecasting

An analogue of `09_skill_forecasting.ipynb`, using **only `trained-pilot-episodes-v1`**
to predict **A01's selected catalogue skill** at **+1, +2 and +5 seconds**. The target
remains the NN-trained A01 pilot; B01 is the scripted observer. Inputs are B01’s
51 observations plus three causal acceleration features (54 features). The corpus
does not contain A01’s observation stream, so the previous 108-feature paired-input
experiment is no longer used. No tactical-corpus files are read or downloaded.

**Run All downloads the required LFS files, prepares data and trains for up to 100 epochs.
Early stopping is enabled by default and can be switched off in the configuration cell.**
Default CUDA acceleration uses **BF16**, **512-window training batches**, **1024-window
evaluation batches**, and a **normalized GPU cache of unique frames**. CPU execution
falls back to FP32 and direct memory-mapped batches. A prefetched CPU data path and
`torch.compile` are optional. The configuration cell exposes all switches. The
new run directory preserves earlier experiments; resume requires the same source
and configuration. It produces inline train/test per-class metrics every epoch and final inline diagnostics
for train, test and validation. No trained results are bundled with this notebook.
Dependencies: Python 3.10+, NumPy, pandas, PyArrow, PyTorch 2.1+, Matplotlib and IPython.
For example, run `%pip install -e ".[ml]"` from the repository root in your notebook environment.
The reported training setup is an **NVIDIA GeForce RTX 5080** (assuming its
standard **16 GB VRAM**) with **PyTorch 2.11.0+cu130** and CUDA 13.1 reported.
`+cu130` identifies a CUDA **13.0** PyTorch build; a driver supporting CUDA 13.1
can run it. These version numbers do not need to match exactly, and no reinstall
is indicated solely by that difference. The runtime table records the actual
PyTorch CUDA runtime, GPU architecture, BF16 availability and a small CUDA kernel check.

The reported CPU is **GenuineIntel, Family 6 Model 186 Stepping 2**, with
**10 physical cores and 16 logical processors**, architecture **AMD64/x86-64**.
AMD64 is the instruction-set name here; the CPU vendor is Intel. The reported
2,500 MHz frequency and 15.5% utilization are snapshots, not throughput benchmarks.
Reported target hardware is shown separately from the environment executing the
notebook, so CPU-only checks performed elsewhere are not attributed to this machine. Prepared frame size is reported below; allow several GB for
raw data, arrays, checkpoints and reports. Histories are gathered from a memory map.

The reference notebook and configuration were inspected at repository revision
`b5e8bb13dda92f9e1070b0cc3f797a7ca8028c60`.
This notebook's companion module is `bvr_behavior_prediction/paired_skill_forecasting.py`.

**MLflow tracking is enabled by default.** Training runs receive unique timestamped
folders and are grouped in the `paired-skill-forecasting` experiment. The dedicated
SQLite store and model/figure artifacts live under `artifacts/paired_skill_forecasting/`.
Set `MLFLOW_TRACKING_URI` for a remote server, or `MLFLOW_ENABLED=False` to disable tracking.
The setup cell prints the command to start the local comparison UI. Resumption uses
`EXISTING_RUN` and reuses the saved MLflow run ID.

Every executable cell starts with IPython's `%%time` magic. Its output reports
**CPU time and elapsed wall time** after the cell finishes, including the setup
cell itself. CUDA work is synchronized before each cell finishes so GPU kernels
are included in elapsed time. Markdown cells contain no executable work and are
not timed. Use an IPython/Jupyter kernel to execute these cells.

On Windows, the notebook kernel must be able to launch Git. The download helper
checks PATH and standard Git for Windows locations. If needed, set
`GIT_EXECUTABLE = r"C:\Program Files\Git\cmd\git.exe"` in the setup cell.
For a Conda-managed install, run `conda install -c conda-forge git git-lfs` in
the notebook's environment, then restart Jupyter so it inherits the updated PATH.
If the raw corpus Parquet files have already been downloaded, set `DOWNLOAD_LFS=False`.


In [5]:
%%time
from pathlib import Path
import json
import os
import platform
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display, Markdown

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "bvr_behavior_prediction/paired_skill_forecasting.py").exists())
sys.path.insert(0, str(ROOT))
from bvr_behavior_prediction.paired_skill_forecasting import (
    DATA_CONTRACT, CORPUS, analyze_trained_corpus, skill_support_table,
    DEFAULT_TRAINING, FEATURES, HORIZONS, HISTORY_STEPS, RAW_HISTORY_STEPS, SPLITS,
    PairedData, prepare_dataset, train_model, final_evaluation, metric_tables,
    read_json, sha256, resolve_precision, benchmark_batches, download_paired_corpora,
)

DATA = ROOT / "artifacts/paired_skill_forecasting/data_trained_pilot_skill_horizon_v2"
RUN_LABEL = "trained-pilot-stratified-bf16"
EXISTING_RUN = None  # Set to an existing run directory for RESUME=True or TRAIN_MODEL=False.
RUN = Path(EXISTING_RUN).expanduser().resolve() if EXISTING_RUN else ROOT / "artifacts/paired_skill_forecasting/runs" / (RUN_LABEL + "_" + pd.Timestamp.now(tz="UTC").strftime("%Y%m%dT%H%M%S%fZ"))
DOWNLOAD_LFS = True
GIT_EXECUTABLE = None  # Or r"C:\Program Files\Git\cmd\git.exe" if Git is absent from the kernel PATH.
PREPARE_DATA = True
TRAIN_MODEL = True
RESUME = False  # Resume the same run; completed/early-stopped runs do not train again.
EARLY_STOPPING = True  # False always runs the full 100 epochs.
if RESUME and EXISTING_RUN is None:
    raise ValueError("Set EXISTING_RUN to the previous run directory before resuming.")
if not TRAIN_MODEL and EXISTING_RUN is None:
    raise ValueError("Set EXISTING_RUN to the completed run directory for a report-only execution.")
MLFLOW_ENABLED = True
MLFLOW_EXPERIMENT = "paired-skill-forecasting"
MLFLOW_DIRECTORY = ROOT / "artifacts/paired_skill_forecasting"
MLFLOW_DIRECTORY.mkdir(parents=True, exist_ok=True)
MLFLOW_TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", f"sqlite:///{(MLFLOW_DIRECTORY / 'mlflow.db').as_posix()}")
MLFLOW_ARTIFACT_ROOT = (MLFLOW_DIRECTORY / "mlflow_artifacts").as_uri()
TRACKING = {"enabled": MLFLOW_ENABLED, "tracking_uri": MLFLOW_TRACKING_URI,
            "experiment": MLFLOW_EXPERIMENT, "run_name": RUN.name,
            "artifact_root": MLFLOW_ARTIFACT_ROOT if MLFLOW_TRACKING_URI.startswith("sqlite:") else None,
            "log_per_class": True, "log_model": True}
from bvr_behavior_prediction.paired_skill_tracking import log_run_artifacts, compare_runs, metric_history
print("Run directory:", RUN)
if MLFLOW_ENABLED and MLFLOW_TRACKING_URI.startswith("sqlite:"):
    print(f'mlflow ui --backend-store-uri "{MLFLOW_TRACKING_URI}" --host 127.0.0.1 --port 5000')
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PRECISION = "bf16"           # CUDA BF16 autocast; CPU/unsupported devices use FP32.
DATA_BACKEND = "gpu_cache"   # Alternatives: "prefetch" (CPU workers), "direct".
BATCH_SIZE = 512             # Benchmark 512 and 1024 below; choose explicitly.
EVALUATION_BATCH_SIZE = 1024
CPU_THREADS = 4
NUM_WORKERS = 2              # Used only by DATA_BACKEND="prefetch"; try 2, then 4.
PREFETCH_FACTOR = 2
COMPILE_MODEL = False        # Opt-in torch.compile; needs a supported Inductor/Triton setup.
COMPILE_MODE = "default"     # Also supports "reduce-overhead" / "max-autotune".
RUN_BATCH_BENCHMARK = DEVICE == "cuda"
BENCHMARK_COMPILATION = False  # True compares eager/compiled; first compilation can be slow.
CONFIG = dict(DEFAULT_TRAINING)
CONFIG.update(early_stopping=EARLY_STOPPING,
              early_stopping_min_epochs=50, early_stopping_patience=10,
              precision=PRECISION, data_backend=DATA_BACKEND,
              batch_size=BATCH_SIZE, evaluation_batch_size=EVALUATION_BATCH_SIZE,
              cpu_threads=CPU_THREADS, num_workers=NUM_WORKERS, prefetch_factor=PREFETCH_FACTOR,
              compile_model=COMPILE_MODEL, compile_mode=COMPILE_MODE)
reported_hardware = {
    "PyTorch": "2.11.0+cu130", "PyTorch CUDA runtime": "13.0",
    "Reported CUDA/driver capability": "13.1", "GPU": "NVIDIA GeForce RTX 5080",
    "GPU memory assumption": "16 GB", "CPU": "Intel64 Family 6 Model 186 Stepping 2, GenuineIntel",
    "Architecture": "AMD64 / x86-64", "Physical cores": 10, "Logical processors": 16,
    "Frequency snapshot": "2500 MHz current/max", "CPU utilization snapshot": "15.5%",
}
display(pd.Series(reported_hardware, name="Reported target training machine"))
runtime = {"PyTorch": str(torch.__version__), "PyTorch CUDA runtime": torch.version.cuda,
           "Selected device": DEVICE, "CPU identifier": platform.processor(),
           "Architecture": platform.machine(), "Logical processors": os.cpu_count(),
           "PyTorch intra-op threads now": torch.get_num_threads(),
           "PyTorch inter-op threads now": torch.get_num_interop_threads(),
           "Requested precision": PRECISION, "Effective precision": resolve_precision(DEVICE, PRECISION),
           "Requested data backend": DATA_BACKEND,
           "Fitting loop intra-op threads": min(CPU_THREADS, os.cpu_count() or 1)}
if DEVICE == "cuda":
    properties = torch.cuda.get_device_properties(0)
    runtime.update({"GPU": properties.name, "GPU memory (GiB)": properties.total_memory / 2**30,
                    "Compute capability": torch.cuda.get_device_capability(0),
                    "Compiled CUDA architectures": torch.cuda.get_arch_list(),
                    "BF16 supported": torch.cuda.is_bf16_supported()})
    # A driver/device query alone does not prove that the installed kernels execute.
    cuda_probe = torch.ones((16, 16), device="cuda")
    cuda_probe_result = cuda_probe @ cuda_probe
    torch.cuda.synchronize()
    runtime["CUDA kernel smoke check"] = "passed"
    del cuda_probe, cuda_probe_result
display(pd.Series(runtime, name="Detected notebook runtime"))
assert CONFIG["epochs"] == 100
assert CONFIG["heads"] == 4 * 1.5 == 6
assert CONFIG["width"] % CONFIG["heads"] == 0
display(pd.Series({**CONFIG, "device": DEVICE, "history_steps": HISTORY_STEPS,
                   "raw_history_steps": RAW_HISTORY_STEPS, "features": len(FEATURES),
                   "horizons_s": HORIZONS}, name="Experiment configuration"))

# Include completion of queued GPU kernels in this cell's elapsed wall time.
if torch.cuda.is_available():
    torch.cuda.synchronize()


Run directory: C:\Users\theon\Trajectory_Classification\artifacts\paired_skill_forecasting\runs\trained-pilot-stratified-bf16_20261105T162742052741Z
mlflow ui --backend-store-uri "sqlite:///C:/Users/theon/Trajectory_Classification/artifacts/paired_skill_forecasting/mlflow.db" --host 127.0.0.1 --port 5000


PyTorch                                                                 2.11.0+cu130
PyTorch CUDA runtime                                                            13.0
Reported CUDA/driver capability                                                 13.1
GPU                                                          NVIDIA GeForce RTX 5080
GPU memory assumption                                                          16 GB
CPU                                Intel64 Family 6 Model 186 Stepping 2, Genuine...
Architecture                                                          AMD64 / x86-64
Physical cores                                                                    10
Logical processors                                                                16
Frequency snapshot                                              2500 MHz current/max
CPU utilization snapshot                                                       15.5%
Name: Reported target training machine, dtype: object

PyTorch                                                               2.11.0+cu130
PyTorch CUDA runtime                                                          13.0
Selected device                                                               cuda
CPU identifier                   Intel64 Family 6 Model 186 Stepping 2, Genuine...
Architecture                                                                 AMD64
Logical processors                                                              16
PyTorch intra-op threads now                                                    10
PyTorch inter-op threads now                                                    10
Requested precision                                                           bf16
Effective precision                                                           bf16
Requested data backend                                                   gpu_cache
Fitting loop intra-op threads                                                    4
GPU 

epochs                             100
seed                                42
batch_size                         512
evaluation_batch_size             1024
learning_rate                   0.0003
weight_decay                      0.01
gradient_clip                      1.0
width                              192
heads                                6
layers                               2
dropout                            0.1
early_stopping                    True
early_stopping_min_epochs           50
early_stopping_patience             10
precision                         bf16
data_backend                 gpu_cache
num_workers                          2
prefetch_factor                      2
cpu_threads                          4
loader_timeout                      60
compile_model                    False
compile_mode                   default
device                            cuda
history_steps                       51
raw_history_steps                   56
features                 

CPU times: total: 15.6 ms
Wall time: 9 ms


## Experimental contract

* **History:** 51 chronological frames at 10 Hz span five seconds. Five extra past
  frames support a backward 0.5-second acceleration estimate. The first origin is
  six seconds, then every second, matching notebook 09. No histories cross episodes.
* **Inputs:** B01's 51 observations plus three backward acceleration features:
  **54 features per frame**, all from `trained-pilot-episodes-v1`.
  Future observations, decisions, pilot IDs, checkpoint IDs and outcomes are excluded.
* **Targets:** the actual A01 skill recorded at each exact future timestamp.
  Missing future decisions are masked separately; there is no forward fill. These
  are catalogue selections, not necessarily distinct executed physical manoeuvres.
* **Architecture:** two Transformer layers, **six attention heads**, dropout 0.1.
  Width increases from 128 to **192** to preserve 32 dimensions per head and allow
  six heads. This changes capacity as well as head count; use an ablation to assess it.
* **Split:** **train/test/validation = 0.4/0.3/0.3**, with whole scenario
  groups kept together across all pilots. Fixed group quotas preserve starting-range
  balance. Seeded stratification balances each skill/horizon's episode presence and
  forecast-window counts toward the same 40/30/30 shares, with equal weight per
  nonzero skill/horizon. This preserves class proportions across partitions; it does
  not downsample common skills to make classes uniform. Rare skills and indivisible
  groups can prevent exact balance; the audit displays achieved shares and absences.
* **Selection:** keep every epoch whose mean **test** NLL strictly improves over
  all previous epochs. The test partition is consequently a model-selection set;
  its final score is selection-biased. Validation is evaluated only after selection
  and provides the held-out estimate. Do not tune against it afterward.
* **Acceleration:** CUDA BF16 autocast for network forward passes; FP32 model weights,
  optimizer state, cross-entropy, probabilities and reporting. Unique frames are
  normalized once on GPU with training-only statistics; labels/anchors are cached
  alongside them. No expanded-window cache is created. Default batches are 512
  for fitting and 1024 for evaluation; epoch batch tails remain included.
* **Training:** up to 100 epochs, with optional early stopping on mean test NLL.
  The first 50 epochs always run; patience counting starts at epoch 51. Ten
  consecutive epochs without a new all-time minimum stop training, so epoch 60
  is the earliest stop. A strict improvement resets the counter. Equal-weight
  mean of the three available horizon cross-entropies, AdamW, gradient clipping,
  train-only scaling.

## Data analysis before splitting

Only the NN-trained pilot corpus is used. An episode exhibits a skill at a horizon
when at least one eligible history has that exact future A01 decision. Counts use
the same six-second first origin, one-second stride, alive-history requirement and
independent missing-label masks as training. Episodes can exhibit several skills,
so column totals need not equal the number of episodes. Zero-count catalogue
skills are retained. Scenario-group counts below identify skills that cannot be
represented in all three splits because fewer than three independent groups exist.


In [6]:
%%time
# Download raw trained-pilot inputs even when a prepared cache already exists:
# the pre-split survey is always rebuilt from these verified sources.
includes = [f"{CORPUS}/{pattern}" for pattern in (
    "episodes.parquet", "observations/*.parquet", "tracks/*.parquet",
    "target_decisions/*.parquet")]
if DOWNLOAD_LFS:
    download_paired_corpora(ROOT, includes, git_executable=GIT_EXECUTABLE)
analysis = analyze_trained_corpus(ROOT)  # No partition assignments are made here.
pre_split_support = skill_support_table(analysis)
print(f"NN-trained pilot episodes: {len(analysis['episodes']):,}")
with pd.option_context("display.max_rows", None):
    display(pre_split_support.pivot(index="skill", columns="horizon_s", values="episodes"))
    display(pre_split_support.pivot(index="skill", columns="horizon_s", values="windows"))
    display(pre_split_support.loc[pre_split_support.scenario_groups.lt(3),
        ["horizon_s", "skill", "episodes", "scenario_groups"]])
# fig, ax = plt.subplots(figsize=(9, max(5, len(analysis["classes"]) * .23)))
# pre_split_support.pivot(index="skill", columns="horizon_s", values="episodes").plot.barh(ax=ax)
# ax.set(xlabel="Episodes with at least one eligible label", ylabel="Skill",
#        title="NN-trained pilot: episode support before splitting")
# ax.legend(title="Forecast horizon (s)")
# plt.tight_layout()
# plt.show()
if torch.cuda.is_available():
    torch.cuda.synchronize()


NN-trained pilot episodes: 3,000


horizon_s,1,2,5
skill,,,
abort_support,129,137,146
accelerate_to_speed,2,2,2
beam_missile_left,22,23,20
beam_missile_right,28,34,42
beam_target_left,1178,1118,1001
beam_target_right,898,907,918
climb_to_altitude,2,2,2
commit,0,0,0
crank_target_left,1689,1628,1393


horizon_s,1,2,5
skill,,,
abort_support,989,1014,1053
accelerate_to_speed,3,3,3
beam_missile_left,65,67,51
beam_missile_right,100,109,145
beam_target_left,13994,13396,11932
beam_target_right,9390,9479,9674
climb_to_altitude,4,4,4
commit,0,0,0
crank_target_left,12825,11748,8938


,horizon_s,skill,episodes,scenario_groups
1,1,accelerate_to_speed,2,2
6,1,climb_to_altitude,2,2
7,1,commit,0,0
13,1,dive_defense,0,0
14,1,drag_missile,0,0
34,2,accelerate_to_speed,2,2
39,2,climb_to_altitude,2,2
40,2,commit,0,0
46,2,dive_defense,0,0
47,2,drag_missile,0,0


CPU times: total: 18.2 s
Wall time: 15.4 s


In [7]:
%%time
# Load a compatible prepared cache even if the optional survey cell was skipped.
if PREPARE_DATA and not (DATA / "manifest.json").exists():
    if globals().get("analysis") is None:
        print("No prepared cache or survey found. Computing skill/horizon counts for stratification.", flush=True)
    prepare_dataset(ROOT, DATA, analysis=globals().get("analysis"))
assert (DATA / "manifest.json").exists(), "Prepare data first or enable PREPARE_DATA."
prepared_manifest = read_json(DATA / "manifest.json")
assert prepared_manifest.get("data_contract") == DATA_CONTRACT, "Use a fresh trained-pilot cache."
source_corpora = {relative.replace("\\", "/") for relative in prepared_manifest["source_manifests"]}
assert source_corpora == {CORPUS}, f"Unexpected raw corpus: {source_corpora}"
data = PairedData(DATA)  # Verifies prepared-file hashes and scenario isolation.
assert data.manifest["split_fractions"] == {"train": .4, "test": .3, "validation": .3}
assert data.manifest["history_steps"] == 51 and data.manifest["raw_history_steps"] == 56
assert data.manifest["features"] == FEATURES
assert data.manifest["horizons_s"] == [1, 2, 5]
for relative, expected in data.manifest["source_manifests"].items():
    assert sha256(ROOT / relative.replace("\\", "/") / "manifest.json") == expected, "Raw corpus provenance changed."
display(pd.DataFrame(data.manifest["counts"]).T)
display(Markdown(f"**Catalogue:** {len(data.classes)} classes. **Device:** {DEVICE}."))

# Include completion of queued GPU kernels in this cell's elapsed wall time.
if torch.cuda.is_available():
    torch.cuda.synchronize()


,groups,episodes,windows,labels_by_horizon
train,120,1200,83811,"[83811, 82928, 80243]"
test,90,900,63153,"[63153, 62469, 60409]"
validation,90,900,62752,"[62752, 62098, 60114]"


**Catalogue:** 33 classes. **Device:** cuda.

CPU times: total: 1.14 s
Wall time: 1.11 s


## Split and label-support audit

Preparation verifies trained-pilot source checksums, episode IDs, frame counts and
exact timestamps. Normalization uses training-history frames only. All three horizons
use independent missing-label masks. Episode presence and forecast-window support
are audited separately; overlapping windows never become separate split units.

The batching and memory table below is calculated from the prepared dataset and
current configuration. Batches contain forecast windows, not whole encounters.
On CUDA, the default `gpu_cache` path copies unique frames in bounded chunks and
normalizes them once with train-only statistics; each batch is gathered on GPU.
The CPU memory map remains unchanged. `prefetch` instead uses a single reusable
worker pool with vectorized batches, pinned memory and a separate CUDA copy stream.
Workers reopen memory maps rather than serializing the dataset (including on
Windows/spawn). Use `NUM_WORKERS=0` to debug that path. `direct` retains the original
CPU gathering path. CPU execution automatically substitutes `direct` for `gpu_cache`.


In [8]:
%%time
assignments = pd.read_csv(DATA / "split_assignments.csv")
display(assignments.groupby("split").agg(
    scenarios=("scenario_group", "nunique"), episodes=("episode_id", "nunique")))
split_support = pd.read_csv(DATA / "skill_support_by_split.csv")
for quantity in ("episodes", "windows"):
    table = split_support.pivot(index=["horizon_s", "skill"], columns="split", values=quantity)
    shares = table.div(table.sum(axis=1).replace(0, np.nan), axis=0)
    display(Markdown(f"**{quantity.title()} by skill/horizon and achieved partition shares** (targets: 0.4/0.3/0.3)"))
    with pd.option_context("display.max_rows", None):
        display(pd.concat({"count": table, "share": shares.round(3)}, axis=1))
missing = split_support.loc[split_support.episodes.eq(0), ["split", "horizon_s", "skill"]]
display(Markdown("**Zero-support split/skill/horizon combinations** (including globally absent skills)"))
display(missing)

# Resource estimates use configured shapes; no additional fitting is performed.
import math
from bvr_behavior_prediction.paired_skill_forecasting import PairedTransformer

with torch.random.fork_rng(devices=[]):
    resource_model = PairedTransformer(num_classes=len(data.classes), **{
        k: CONFIG[k] for k in ("width", "heads", "layers", "dropout")})
    parameter_count = sum(p.numel() for p in resource_model.parameters())
del resource_model

window_bytes = HISTORY_STEPS * len(FEATURES) * np.dtype(np.float32).itemsize
resource_rows = [
    ("Parameters", parameter_count, "parameters"),
    ("FP32 model weights", parameter_count * 4 / 2**20, "MiB"),
    ("FP32 weights + gradients + two Adam states", parameter_count * 16 / 2**20, "MiB"),
    ("Raw feature memory-map file", data.features.nbytes / 2**30, "GiB (not necessarily RSS)"),
    ("GPU frame/anchor/target cache estimate", (data.features.nbytes + data.anchors.nbytes
                                             + data.targets.nbytes) / 2**30, "GiB"),
    ("Expanded windows if all were cached", len(data.targets) * window_bytes / 2**30, "GiB"),
    ("Training batch input", CONFIG["batch_size"] * window_bytes / 2**20, "MiB"),
    ("Evaluation batch input", CONFIG["evaluation_batch_size"] * window_bytes / 2**20, "MiB"),
    ("Optimizer batches per epoch", math.ceil(len(data.indices["train"]) / CONFIG["batch_size"]), "batches"),
]
for split in ("train", "test"):
    count = len(data.indices[split])
    resource_rows.append((f"{split} evaluation batches per epoch",
                          math.ceil(count / CONFIG["evaluation_batch_size"]), "batches"))
    resource_rows.append((f"{split} evaluation probabilities",
                          count * len(HORIZONS) * len(data.classes) * 4 / 2**20, "MiB"))
display(pd.DataFrame(resource_rows, columns=["Quantity", "Value", "Unit"]).round(3))
print("Model activations, framework workspaces, allocator reserves, data-page cache and")
print("temporary arrays are additional. Measure CUDA peak allocated/reserved memory on the target GPU.")

# Disposable training-only benchmark: no test/validation access and no run/checkpoint writes.
# RNG states are restored afterward. This does not change BATCH_SIZE automatically.
if RUN_BATCH_BENCHMARK:
    batch_benchmark = benchmark_batches(data, CONFIG, DEVICE, batch_sizes=(512, 1024),
                                        compare_compilation=BENCHMARK_COMPILATION)
    display(batch_benchmark.round(4))
    benchmark_dir = ROOT / "artifacts/paired_skill_forecasting/benchmarks"
    benchmark_dir.mkdir(parents=True, exist_ok=True)
    benchmark_path = benchmark_dir / ("batches_" + pd.Timestamp.now(tz="UTC").strftime("%Y%m%dT%H%M%S%fZ") + ".csv")
    batch_benchmark.to_csv(benchmark_path, index=False)
    print("Saved benchmark:", benchmark_path)
    TRACKING["benchmark_path"] = str(benchmark_path)
else:
    print("Batch benchmark skipped. Enable RUN_BATCH_BENCHMARK to run it explicitly.")

# Include completion of queued GPU kernels in this cell's elapsed wall time.
if torch.cuda.is_available():
    torch.cuda.synchronize()


,scenarios,episodes
split,,
test,90,900
train,120,1200
validation,90,900


**Episodes by skill/horizon and achieved partition shares** (targets: 0.4/0.3/0.3)

count                   share                  
split                          test train validation   test  train validation
horizon_s skill                                                              
1         abort_support          39    53         37  0.302  0.411      0.287
          accelerate_to_speed     0     1          1  0.000  0.500      0.500
          beam_missile_left       7     8          7  0.318  0.364      0.318
          beam_missile_right      8    11          9  0.286  0.393      0.321
          beam_target_left      357   471        350  0.303  0.400      0.297
          beam_target_right     265   364        269  0.295  0.405      0.300
          climb_to_altitude       1     1          0  0.500  0.500      0.000
          commit                  0     0          0    NaN    NaN        NaN
          crank_target_left     504   682        503  0.298  0.404      0.298
          crank_target_right    110   146        110  0.301  0.399      0.301
          decelerate_to_speed    41    56         41  0.297  0.406      0.297
          defensive_reversal      1     2          1  0.250  0.500      0.250
          descend_to_altitude    51    68         54  0.295  0.393      0.312
          dive_defense            0     0          0    NaN    NaN        NaN
          drag_missile            0     0          0    NaN    NaN        NaN
          extend                178   236        175  0.302  0.401      0.297
          lag_pursuit           303   408        298  0.300  0.404      0.295
          last_ditch_break        1     1          1  0.333  0.333      0.333
          launch                 41    56         43  0.293  0.400      0.307
          lead_pursuit          196   256        194  0.303  0.396      0.300
          lock_target             1     2          1  0.250  0.500      0.250
          maintain_heading       13    15         12  0.325  0.375      0.300
          notch_left             55    74         54  0.301  0.404      0.295
          notch_right            40    52         42  0.299  0.388      0.313
          preemptive_crank        1     2          2  0.200  0.400      0.400
          pursue_target         200   264        198  0.302  0.399      0.299
          recommit              309   407        308  0.302  0.397      0.301
          search                  6     9          6  0.286  0.429      0.286
          secondary_shot        359   474        356  0.302  0.399      0.299
          short_range_attack     66    89         64  0.301  0.406      0.292
          support_missile        14    18         14  0.304  0.391      0.304
          turn_cold             279   372        277  0.301  0.401      0.298
          turn_to_heading        18    24         18  0.300  0.400      0.300
2         abort_support          42    55         40  0.307  0.401      0.292
          accelerate_to_speed     0     1          1  0.000  0.500      0.500
          beam_missile_left       7     9          7  0.304  0.391      0.304
          beam_missile_right     10    14         10  0.294  0.412      0.294
          beam_target_left      334   449        335  0.299  0.402      0.300
          beam_target_right     270   366        271  0.298  0.404      0.299
          climb_to_altitude       1     1          0  0.500  0.500      0.000
          commit                  0     0          0    NaN    NaN        NaN
          crank_target_left     490   658        480  0.301  0.404      0.295
          crank_target_right    113   152        113  0.299  0.402      0.299
          decelerate_to_speed    43    56         42  0.305  0.397      0.298
          defensive_reversal      1     2          2  0.200  0.400      0.400
          descend_to_altitude    52    68         54  0.299  0.391      0.310
          dive_defense            0     0          0    NaN    NaN        NaN
          drag_missile            0     0          0    NaN    NaN        NaN
          extend                176   234        179

**Windows by skill/horizon and achieved partition shares** (targets: 0.4/0.3/0.3)

count                    share         \
split                           test  train validation   test  train   
horizon_s skill                                                        
1         abort_support          297    397        295  0.300  0.401   
          accelerate_to_speed      0      2          1  0.000  0.667   
          beam_missile_left       18     27         20  0.277  0.415   
          beam_missile_right      31     38         31  0.310  0.380   
          beam_target_left      4195   5603       4196  0.300  0.400   
          beam_target_right     2847   3744       2799  0.303  0.399   
          climb_to_altitude        1      3          0  0.250  0.750   
          commit                   0      0          0    NaN    NaN   
          crank_target_left     3835   5135       3855  0.299  0.400   
          crank_target_right    1761   2353       1746  0.301  0.402   
          decelerate_to_speed    300    390        309  0.300  0.390   
          defensive_reversal       6      3          1  0.600  0.300   
          descend_to_altitude    496    612        472  0.314  0.387   
          dive_defense             0      0          0    NaN    NaN   
          drag_missile             0      0          0    NaN    NaN   
          extend                5391   7267       5400  0.299  0.402   
          lag_pursuit          11369  14787      10856  0.307  0.400   
          last_ditch_break         2      3          1  0.333  0.500   
          launch                 273    375        277  0.295  0.405   
          lead_pursuit          3121   4184       3126  0.299  0.401   
          lock_target              3      2          3  0.375  0.250   
          maintain_heading        47     64         44  0.303  0.413   
          notch_left             351    459        349  0.303  0.396   
          notch_right            162    217        159  0.301  0.403   
          preemptive_crank         2      2          2  0.333  0.333   
          pursue_target         4579   6082       4595  0.300  0.399   
          recommit              9035  12066       9147  0.299  0.399   
          search                  17     23         17  0.298  0.404   
          secondary_shot        3075   4070       3108  0.300  0.397   
          short_range_attack     379    498        385  0.300  0.395   
          support_missile         93    121         95  0.301  0.392   
          turn_cold            11378  15167      11365  0.300  0.400   
          turn_to_heading         89    117         98  0.293  0.385   
2         abort_support          304    405        305  0.300  0.399   
          accelerate_to_speed      0      2          1  0.000  0.667   
          beam_missile_left       19     28         20  0.284  0.418   
          beam_missile_right      33     43         33  0.303  0.394   
          beam_target_left      4015   5356       4025  0.300  0.400   
          beam_target_right     2871   3779       2829  0.303  0.399   
          climb_to_altitude        1      3          0  0.250  0.750   
          commit                   0      0          0    NaN    NaN   
          crank_target_left     3512   4699       3537  0.299  0.400   
          crank_target_right    1769   2364       1752  0.301  0.402   
          decelerate_to_speed    302    394        309  0.300  0.392   
          defensive_reversal       6      3          2  0.545  0.273   
          descend_to_altitude    479    604        458  0.311  0.392   
          dive_defense             0      0          0    NaN    NaN   
          drag_missile             0      0          0    NaN    NaN   
          extend                5377   7248       5379  0.299  0.403   
          lag_pursuit          11324  14711      10833  0.307  0.399   
          last_ditch_break         2      3          1  0.333  0.500   
          launch                 274    373        275  0.297  0.405   
          lead_pursuit          3104   4169       3122  0.299  0.401   

**Zero-support split/skill/horizon combinations** (including globally absent skills)

,split,horizon_s,skill
7,train,1,commit
13,train,1,dive_defense
14,train,1,drag_missile
40,train,2,commit
46,train,2,dive_defense
47,train,2,drag_missile
73,train,5,commit
79,train,5,dive_defense
80,train,5,drag_missile
100,test,1,accelerate_to_speed


,Quantity,Value,Unit
0,Parameters,966627.000,parameters
1,FP32 model weights,3.687,MiB
2,FP32 weights + gradients + two Adam states,14.750,MiB
3,Raw feature memory-map file,0.514,GiB (not necessarily RSS)
4,GPU frame/anchor/target cache estimate,0.520,GiB
5,Expanded windows if all were cached,2.152,GiB
6,Training batch input,5.379,MiB
7,Evaluation batch input,10.758,MiB
8,Optimizer batches per epoch,164.000,batches
9,train evaluation batches per epoch,82.000,batches


Model activations, framework workspaces, allocator reserves, data-page cache and
temporary arrays are additional. Measure CUDA peak allocated/reserved memory on the target GPU.


,batch_size,compiled,status,precision,backend,frame_cache_GiB,warmup_seconds,median_step_seconds,windows_per_second,peak_allocated_MiB,peak_reserved_MiB
0,512,False,ok,bf16,gpu_cache,0.5203,0.3973,0.0135,37812.4884,1096.2266,1238.0
1,1024,False,ok,bf16,gpu_cache,0.5203,0.0663,0.0300,34178.4489,1588.6572,1756.0


Saved benchmark: C:\Users\theon\Trajectory_Classification\artifacts\paired_skill_forecasting\benchmarks\batches_20261105T162816455677Z.csv
CPU times: total: 5.2 s
Wall time: 2.27 s


## Train for up to 100 epochs

Each epoch ends with a fresh **evaluation-mode** pass over train and test, using
the same updated checkpoint, without dropout. Each inline table contains per-class
F1, support and **class-conditional NLL** (mean cross-entropy among examples whose
true label is that class) at all three horizons. Unsupported classes have zero F1
and undefined loss. Macro-F1 includes all catalogue classes, with undefined F1 set
to zero, matching notebook 09. Supported macro-F1 is also recorded.

Outputs are retained for every epoch; there is no clearing of previous output.
`epochs/` stores full CSVs; `checkpoints/epoch_NNN.pt` retains **every new minimum**
of mean test NLL. `best.pt` selects the lowest-loss epoch and `last.pt` supports
resumption, including optimizer, RNG and early-stopping counter states.
With `EARLY_STOPPING=True`, counting begins after epoch 50; ten consecutive
non-improving test-loss epochs stop training. Set it to `False` for all 100 epochs.
`training_status.json` records whether training completed or stopped early.
Resuming an early-stopped run returns its history without fitting more epochs. A nonempty run directory requires
`RESUME=True` or a fresh `RUN` path. GPU/CPU floating-point results can differ. Effective precision, backend, CPU
thread count and cache size are recorded in `run.json`. Changing precision, batch
size, compilation or data backend requires a fresh run directory.

The batch benchmark uses disposable models and **training data only**, preserves
RNG state, includes gathering/transfers/forward/backward/AdamW in each step, and
reports measured throughput plus peak allocated/reserved VRAM. Cold/warmup time
is separate. It never adjusts the learning rate or selects a checkpoint. Set
`BATCH_SIZE=1024` only after comparing the benchmark and memory headroom; larger
batches change the number of optimizer updates per epoch. `BENCHMARK_COMPILATION=True`
compares eager and compiled variants. Compilation is optional and can require
additional backend support on Windows; failures should be resolved or compilation
disabled, rather than assuming an unmeasured speedup. Checkpoints always save
unwrapped FP32 weights and can be loaded with or without compilation.

The `%%time` output at the bottom of the fitting cell measures the entire fitting
cell, including checkpoint writes and inline rendering. The existing per-epoch
`seconds` field measures fitting and train/test evaluation before checkpoint/table
and MLflow I/O; it should not be mistaken for total per-epoch wall time. A phase profiler is
recommended before changing the implementation for throughput.

### MLflow run/model comparison

Configuration, dataset/source hashes and actual runtime precision/backend are tracked.
Per-epoch train/test metrics are logged both as horizon means and by horizon; per-class
F1, class-conditional NLL and support are included by default. Unsupported-class NLL
is omitted rather than logged as zero. Set `TRACKING["log_per_class"]=False` to reduce
metric logging; the complete per-class CSV artifacts are still retained.

Every local improving checkpoint remains available. MLflow stores the selected
`best.pt`, all epoch/final metric CSVs, source/data metadata, and final PNG figures;
it does not copy the large raw corpora, prepared feature arrays or prediction NPZs.
The selected network is also exported as a standard **MLflow PyTorch model** at
`runs:/<run_id>/model`, with a signature, packaged source, normalization statistics,
feature ordering, classes and horizons. It accepts **already-normalized (B,51,54)
feature histories** and returns **(B,3,C) logits**. Weights are FP32; use the recorded
CUDA autocast precision when reproducing BF16 evaluation. No model registry deployment
or serving endpoint is created.

Use the MLflow UI to select runs and compare parameters/metrics. The final notebook
section provides a comparison table and loss curves filtered to the **same prepared
dataset hash**. Test loss selects checkpoints; validation remains a final descriptive
score. A new experiment needs a new untouched evaluation set if those validation
scores are used to guide further tuning.

A failed/interrupted attempt is marked FAILED/KILLED; resumption reuses the run ID
and backfills any completed local epochs not yet tracked. Model checkpoints remain
portable and local even if an MLflow write fails. Tracking uses a client scoped to
the specified URI and does not replace an unrelated fluent MLflow active run.



In [9]:
%%time
def epoch_output(row, summary, per_class):
    marker = " — saved improving checkpoint" if row["improved"] else ""
    display(Markdown(f"### Epoch {row['epoch']:03d}/{CONFIG['epochs']}{marker}"))
    display(summary[["split", "horizon_s", "nll", "accuracy", "macro_f1",
                     "supported_macro_f1", "weighted_f1"]].round(4))
    table = per_class.pivot(index=["horizon_s", "skill"], columns="split",
                            values=["class_nll", "f1", "support"])
    with pd.option_context("display.max_rows", None, "display.max_columns", None):
        display(table.round(4))
    print(f"Mean train NLL: {row['train_nll']:.6f}; "
          f"mean test NLL: {row['test_nll']:.6f}; {row['seconds']:.1f} seconds", flush=True)
    if CONFIG["early_stopping"] and row["epoch"] > CONFIG["early_stopping_min_epochs"]:
        print(f"Early-stopping patience: {row['epochs_without_improvement']}/"
              f"{CONFIG['early_stopping_patience']}", flush=True)

if TRAIN_MODEL:
    history = train_model(data, RUN, CONFIG, callback=epoch_output, device=DEVICE, resume=RESUME, tracking=TRACKING)
else:
    assert (RUN / "history.csv").exists(), "Enable TRAIN_MODEL or select a completed run."
    history = pd.read_csv(RUN / "history.csv")
display(history.tail())

# Include completion of queued GPU kernels in this cell's elapsed wall time.
if torch.cuda.is_available():
    torch.cuda.synchronize()


MLflow run: 5dbf3303f30546ccb93b3b04f152f947 (paired-skill-forecasting)
Runtime: cuda, bf16, gpu_cache, batch 512, evaluation batch 1024, frame cache 0.520 GiB


### Epoch 001/100 — saved improving checkpoint

,split,horizon_s,nll,accuracy,macro_f1,supported_macro_f1,weighted_f1
0,train,1,1.4834,0.4632,0.1619,0.1781,0.4469
1,train,2,1.4811,0.4624,0.1642,0.1806,0.4462
2,train,5,1.5478,0.4451,0.1508,0.1659,0.4285
3,test,1,1.5740,0.4379,0.1574,0.1791,0.4236
4,test,2,1.5693,0.4370,0.1572,0.1789,0.4225
5,test,5,1.6323,0.4201,0.1444,0.1643,0.4068


class_nll              f1          support  \
split                              test   train    test   train     test   
horizon_s skill                                                            
1         abort_support          4.2230  4.1751  0.0000  0.0000    297.0   
          accelerate_to_speed       NaN  5.8387  0.0000  0.0000      0.0   
          beam_missile_left      6.5644  6.8864  0.0000  0.0000     18.0   
          beam_missile_right     7.2068  5.7856  0.0000  0.0000     31.0   
          beam_target_left       1.4556  1.3527  0.4843  0.4975   4195.0   
          beam_target_right      1.5484  1.4475  0.4468  0.4605   2847.0   
          climb_to_altitude      6.6658  6.8898  0.0000  0.0000      1.0   
          commit                    NaN     NaN  0.0000  0.0000      0.0   
          crank_target_left      0.9415  0.9588  0.5707  0.5780   3835.0   
          crank_target_right     1.8488  1.9097  0.4303  0.4224   1761.0   
          decelerate_to_speed    3.1803  3.1140  0.0000  0.0000    300.0   
          defensive_reversal     6.7256  6.0310  0.0000  0.0000      6.0   
          descend_to_altitude    1.6988  1.5948  0.5083  0.4713    496.0   
          dive_defense              NaN     NaN  0.0000  0.0000      0.0   
          drag_missile              NaN     NaN  0.0000  0.0000      0.0   
          extend                 1.6083  1.5947  0.3759  0.3734   5391.0   
          lag_pursuit            1.3925  1.3265  0.4652  0.4955  11369.0   
          last_ditch_break       5.9569  5.9653  0.0000  0.0000      2.0   
          launch                 3.9631  3.6166  0.0000  0.0000    273.0   
          lead_pursuit           2.4281  2.3509  0.1936  0.1509   3121.0   
          lock_target            6.4052  6.2907  0.0000  0.0000      3.0   
          maintain_heading       4.7935  4.4215  0.0000  0.0000     47.0   
          notch_left             3.3788  3.2440  0.0000  0.0000    351.0   
          notch_right            4.0099  3.7410  0.0000  0.0000    162.0   
          preemptive_crank       6.3641  7.5696  0.0000  0.0000      2.0   
          pursue_target          1.8691  1.7771  0.3416  0.4075   4579.0   
          recommit               1.3072  1.1954  0.5039  0.5248   9035.0   
          search                 6.4878  6.4591  0.0000  0.0000     17.0   
          secondary_shot         1.5492  1.3609  0.4472  0.4836   3075.0   
          short_range_attack     2.7250  2.7117  0.0000  0.0000    379.0   
          support_missile        4.5995  4.5631  0.0000  0.0000     93.0   
          turn_cold              1.4205  1.2644  0.4254  0.4776  11378.0   
          turn_to_heading        5.0240  4.9415  0.0000  0.0000     89.0   
2         abort_support          4.0989  4.0521  0.0000  0.0000    304.0   
          accelerate_to_speed       NaN  6.3970  0.0000  0.0000      0.0   
          beam_missile_left      6.5853  7.0209  0.0000  0.0000     19.0   
          beam_missile_right     6.7546  5.6031  0.0000  0.0000     33.0   
          beam_target_left       1.3699  1.2626  0.4943  0.5087   4015.0   
          beam_target_right      1.5115  1.3816  0.4486  0.4744   2871.0   
          climb_to_altitude      6.1269  6.4250  0.0000  0.0000      1.0   
          commit                    NaN     NaN  0.0000  0.0000      0.0   
          crank_target_left      1.0267  1.0613  0.5615  0.5639   3512.0   
          crank_target_right     1.8501  1.9166  0.4361  0.4457   1769.0   
          decelerate_to_speed    2.9897  2.8970  0.0000  0.0000    302.0   
          defensive_reversal     7.1531  6.0987  0.0000  0.0000      6.0   
          descend_to_altitude    1.9482  1.9408  0.4787  0.4586    479.0   
          dive_defense              NaN     NaN  0.0000  0.0000      0.0   
          drag_missile              NaN     NaN  0.0000  0.0000      0.0   
          extend                 1.5845  1.5689  0.3774  0.3783   5377.0   
          lag_pursuit            1.3647  1.3074  0.4695  0.4890  11324.0   
  

Mean train NLL: 1.504083; mean test NLL: 1.591842; 8.6 seconds


### Epoch 002/100 — saved improving checkpoint

... (234 outputs left out)

Mean train NLL: 0.410544; mean test NLL: 2.028565; 3.6 seconds
Early-stopping patience: 10/10
Early stopping after epoch 60: 10 consecutive epochs without improved test loss after the first 50 epochs. Best checkpoint: epoch 9.


W1105 16:35:44.980000 13568 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


,epoch,train_nll,test_nll,train_macro_f1,test_macro_f1,improved,epochs_without_improvement,early_stopped,seconds
55,56,0.426609,1.938096,0.564979,0.349787,False,6,False,6.488060
56,57,0.434499,1.957838,0.554666,0.339821,False,7,False,3.618788
57,58,0.420521,1.933471,0.559136,0.338856,False,8,False,3.512258
58,59,0.422815,2.014244,0.555414,0.336457,False,9,False,6.462827
59,60,0.410544,2.028565,0.565229,0.338100,False,10,True,3.611450


CPU times: total: 4min 17s
Wall time: 7min 30s


## Final results from the selected checkpoint

Training can conclude at the early-stopping epoch or at the 100-epoch limit.
The selected checkpoint may precede the final fitted epoch. All final diagnostics below use that
single checkpoint on **train, test and validation**. Validation is first evaluated
here, after selection. A training-frequency baseline provides context; its class
probabilities are fitted only on training labels, separately by horizon.

In [10]:
%%time
status = read_json(RUN / "training_status.json")
assert status["status"] in {"completed", "early_stopped"}, "Training has not concluded."
assert status["completed_epochs"] == len(history), "Training status/history mismatch."
assert read_json(RUN / "run.json")["training"] == CONFIG, "Run configuration differs."
display(Markdown(f"**Training status: {status['status']}; "
                 f"epochs fitted: {len(history)}/{CONFIG['epochs']}.**"))
best_epoch, final_metrics, final_per_class, final_results = final_evaluation(data, RUN, DEVICE, tracking=TRACKING)
display(Markdown(f"**Selected epoch: {best_epoch}.** Test selected this checkpoint; "
                 "validation was excluded from selection."))
display(final_metrics.round(4))

baseline_rows = []
for split in SPLITS:
    y = data.targets[data.indices[split]]
    probabilities = np.zeros((len(y), 3, len(data.classes)), np.float32)
    for h in range(3):
        train_labels = data.targets[data.indices["train"], h]
        counts = np.bincount(train_labels[train_labels >= 0], minlength=len(data.classes))
        probabilities[:, h] = (counts + 1) / (counts.sum() + len(data.classes))
    baseline, _, _ = metric_tables(y, probabilities, data.classes, split)
    baseline_rows.append(baseline)
baseline_metrics = pd.concat(baseline_rows, ignore_index=True)
baseline_metrics.to_csv(RUN / "training_frequency_baseline.csv", index=False)
display(baseline_metrics.assign(model="Training frequency").round(4))

# Include completion of queued GPU kernels in this cell's elapsed wall time.
if torch.cuda.is_available():
    torch.cuda.synchronize()


**Training status: early_stopped; epochs fitted: 60/100.**

MLflow run: 5dbf3303f30546ccb93b3b04f152f947 (paired-skill-forecasting)


**Selected epoch: 9.** Test selected this checkpoint; validation was excluded from selection.

,split,horizon_s,count,nll,accuracy,macro_f1,supported_macro_f1,weighted_f1
0,train,1,83811,0.8571,0.6700,0.3311,0.3642,0.6673
1,train,2,82928,0.8601,0.6675,0.3244,0.3568,0.6645
2,train,5,80243,0.9510,0.6384,0.3018,0.3320,0.6354
3,test,1,63153,1.3280,0.5489,0.2563,0.2917,0.5446
4,test,2,62469,1.3464,0.5416,0.2503,0.2849,0.5370
5,test,5,60409,1.4077,0.5192,0.2322,0.2642,0.5147
6,validation,1,62752,1.3766,0.5388,0.2432,0.2768,0.5341
7,validation,2,62098,1.3998,0.5335,0.2398,0.2729,0.5289
8,validation,5,60114,1.4593,0.5100,0.2242,0.2551,0.5054


,split,horizon_s,count,nll,accuracy,macro_f1,supported_macro_f1,weighted_f1,model
0,train,1,83811,2.4037,0.1810,0.0093,0.0102,0.0555,Training frequency
1,train,2,82928,2.4027,0.1824,0.0093,0.0103,0.0563,Training frequency
2,train,5,80243,2.3943,0.1868,0.0095,0.0105,0.0588,Training frequency
3,test,1,63153,2.4035,0.1802,0.0093,0.0105,0.0550,Training frequency
4,test,2,62469,2.4012,0.1816,0.0093,0.0106,0.0558,Training frequency
5,test,5,60409,2.3898,0.1858,0.0095,0.0108,0.0582,Training frequency
6,validation,1,62752,2.4075,0.1811,0.0093,0.0106,0.0555,Training frequency
7,validation,2,62098,2.4049,0.1826,0.0094,0.0107,0.0564,Training frequency
8,validation,5,60114,2.3946,0.1870,0.0095,0.0109,0.0589,Training frequency


CPU times: total: 6.08 s
Wall time: 7.39 s


In [ ]:
%%time
DIAGNOSTICS = RUN / "diagnostics"
DIAGNOSTICS.mkdir(exist_ok=True)

def show_figure(fig, name):
    fig.tight_layout()
    fig.savefig(DIAGNOSTICS / f"{name}.png", dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for split in ["train", "test"]:
    axes[0].plot(history.epoch, history[f"{split}_nll"], label=split)
    axes[1].plot(history.epoch, history[f"{split}_macro_f1"], label=split)
for ax in axes:
    ax.axvline(best_epoch, color="black", linestyle=":", label="Selected epoch")
    ax.set_xlabel("Epoch")
    ax.legend()
axes[0].set(title="Mean horizon loss", ylabel="NLL / cross-entropy")
axes[1].set(title="Mean horizon catalogue macro-F1", ylabel="F1", ylim=(0, 1))
show_figure(fig, "learning_curves")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, measure in zip(axes, ["nll", "accuracy", "macro_f1"]):
    final_metrics.pivot(index="horizon_s", columns="split", values=measure)[list(SPLITS)].plot(
        kind="bar", ax=ax, rot=0)
    ax.set(title=f"Best checkpoint: {measure}", xlabel="Forecast horizon (s)")
    if measure != "nll": ax.set_ylim(0, 1)
show_figure(fig, "final_split_comparison")

# Include completion of queued GPU kernels in this cell's elapsed wall time.
if torch.cuda.is_available():
    torch.cuda.synchronize()


In [ ]:
%%time
# F1 and support for all catalogue labels, split by horizon.
fig, axes = plt.subplots(2, 3, figsize=(20, 14), sharey=True)
for h, horizon in enumerate(HORIZONS):
    subset = final_per_class.loc[final_per_class.horizon_s.eq(horizon)]
    f1 = subset.pivot(index="skill", columns="split", values="f1").reindex(
        index=data.classes, columns=SPLITS)
    counts = subset.pivot(index="skill", columns="split", values="support").reindex(
        index=data.classes, columns=SPLITS)
    image = axes[0, h].imshow(f1, vmin=0, vmax=1, aspect="auto", cmap="viridis")
    count_image = axes[1, h].imshow(np.log1p(counts), aspect="auto", cmap="magma")
    for ax in axes[:, h]:
        ax.set_xticks(range(3), SPLITS)
        ax.set_yticks(range(len(data.classes)), data.classes, fontsize=7)
    axes[0, h].set_title(f"+{horizon}s per-class F1")
    axes[1, h].set_title(f"+{horizon}s log(1 + support)")
    fig.colorbar(image, ax=axes[0, h], fraction=.04)
    fig.colorbar(count_image, ax=axes[1, h], fraction=.04)
show_figure(fig, "per_class_f1_and_support")

fig, axes = plt.subplots(3, 3, figsize=(24, 23))
for row, split in enumerate(SPLITS):
    for h, horizon in enumerate(HORIZONS):
        cm = final_results[split]["confusion"][h]
        totals = cm.sum(1, keepdims=True)
        normalized = np.divide(cm, totals, out=np.zeros_like(cm, float), where=totals > 0)
        ax = axes[row, h]
        ax.imshow(normalized, vmin=0, vmax=1, cmap="Blues")
        ax.set_xticks(range(len(data.classes)), data.classes, rotation=90, fontsize=6)
        ax.set_yticks(range(len(data.classes)), data.classes, fontsize=6)
        ax.set(title=f"{split}, +{horizon}s", xlabel="Predicted skill", ylabel="True skill")
show_figure(fig, "confusion_matrices")

# Include completion of queued GPU kernels in this cell's elapsed wall time.
if torch.cuda.is_available():
    torch.cuda.synchronize()


In [ ]:
%%time
# Probability calibration: each point averages one confidence bin.
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for h, horizon in enumerate(HORIZONS):
    axes[h].plot([0, 1], [0, 1], "k:", label="Perfect calibration")
    for split in SPLITS:
        result = final_results[split]
        valid = result["targets"][:, h] >= 0
        p = result["probabilities"][valid, h]
        y = result["targets"][valid, h]
        confidence, correct = p.max(-1), p.argmax(-1) == y
        bins = np.minimum((confidence * 10).astype(int), 9)
        points = [(confidence[bins == b].mean(), correct[bins == b].mean())
                  for b in range(10) if (bins == b).any()]
        axes[h].plot(*np.asarray(points).T, marker="o", label=split)
    axes[h].set(title=f"+{horizon}s reliability", xlabel="Mean confidence",
                ylabel="Observed accuracy", xlim=(0, 1), ylim=(0, 1))
    axes[h].legend()
show_figure(fig, "reliability")

with pd.option_context("display.max_rows", None):
    display(final_per_class.pivot(index=["horizon_s", "skill"], columns="split",
                                  values=["class_nll", "f1", "support"]).round(4))
print("Saved metrics, predictions, checkpoints and PNG diagnostics to:", RUN)
if MLFLOW_ENABLED:
    log_run_artifacts(RUN)  # Attach final figures and baseline tables to the same run.
if (RUN / "mlflow.json").exists():
    print("MLflow run ID:", read_json(RUN / "mlflow.json")["run_id"])

# Include completion of queued GPU kernels in this cell's elapsed wall time.
if torch.cuda.is_available():
    torch.cuda.synchronize()


## Recommendations for further performance improvements

These are proposed experiments, not claims of measured gains. Use the selection
partition for tuning; keep a new independent evaluation set for subsequent work
once the final validation results have been inspected.

1. **Measure the value of pairing.** A future paired-input experiment would require an additional corpus. Compare B01-only, A01-only and paired inputs
   under the same 40/30/30 scenario split. Compare 4 heads at width 128, 4 heads at
   width 192 and 6 heads at width 192 to separate width from head-count effects.
   Paired telemetry can expose A01's own missile state; deploy it only where both
   viewpoints can actually be supplied with the same timing and latency.
2. **Improve rare-class recall.** Inspect per-class F1 alongside support and confusion.
   Try capped inverse-square-root class weights, focal loss or balanced sampling
   fitted only on training labels. Compare unweighted NLL, supported macro-F1 and
   calibration as well as accuracy; absent classes cannot be learned from reweighting.
3. **Model the two viewpoints explicitly.** Compare concatenation with separate
   encoders and cross-attention or aircraft/role embeddings, keeping temporal
   alignment exact. Add shared-world relative geometry only from causal observations.
4. **Reduce overfitting if the curves separate.** Try learning-rate warmup and cosine
   decay, dropout/weight-decay tuning, label smoothing and smaller encoders. Tune
   within a fixed search budget; report variability across several training seeds.
5. **Handle skill changes and equivalent controls.** Diagnose transition windows
   separately from stable labels. Consider hierarchical skill families or a
   multi-task skill/continuous-parameter objective, while keeping target decisions
   out of the feature stream. Distinguish catalogue prediction from manoeuvre recognition.
6. **Calibrate and quantify uncertainty.** If reliability curves show overconfidence,
   fit temperature scaling on a separate calibration subset of the selection data.
   Estimate uncertainty by resampling whole scenario groups, not overlapping windows.
   Report per-pilot performance and leave-one-pilot-out tests for unseen-policy claims.
7. **Stress-test realistic access.** Evaluate missing/delayed viewpoints, sensor noise,
   track loss and new geometries. These corpora provide idealized exact tracks.
   Use structured viewpoint dropout only after defining how missing telemetry is
   represented, and validate it on a matching deployment task.

The loss checkpoint rule follows the requested test-based selection. For a future
experiment, conventional validation-based selection with a fully untouched test set
would make the roles easier to interpret. Do not rename these saved partitions
mid-experiment or compare their scores as if they used notebook 09's original split.

## Training throughput and memory

The dynamic audit above calculates model parameters, feature storage, batches per
epoch and input memory from this trained-pilot-only dataset. Previous paired-input
corpus timings, memory figures and batch counts do not apply to this experiment.
Benchmark the current configuration on the target GPU before estimating runtime.


## Compare tracked runs

This view uses runs in the configured experiment with the same prepared-data hash.
It displays final held-out scores alongside test-selected model quality. Curves use
training/test history only. The saved model URI can be loaded with
`mlflow.pytorch.load_model` after setting the matching tracking URI, or by downloading
its model artifact through `MlflowClient`.


In [ ]:
%%time
if MLFLOW_ENABLED:
    comparison = compare_runs(TRACKING, sha256(DATA / "manifest.json"))
    if comparison.empty:
        print("No tracked runs for this prepared dataset yet.")
    else:
        with pd.option_context("display.max_columns", None):
            display(comparison.sort_values("best_test_nll", na_position="last"))
        comparison.to_csv(RUN / "mlflow_comparison.csv", index=False)
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        for row in comparison.head(5).itertuples():
            for ax, split in zip(axes, ("train", "test")):
                curve = metric_history(TRACKING, row.run_id, f"epoch/{split}/nll")
                if not curve.empty:
                    ax.plot(curve.epoch, curve.value, label=str(row.name)[:55])
                ax.set(title=f"Tracked {split} loss", xlabel="Epoch", ylabel="NLL")
        for ax in axes:
            handles, labels = ax.get_legend_handles_labels()
            if handles:
                ax.legend(fontsize=7)
        show_figure(fig, "mlflow_run_comparison")
        log_run_artifacts(RUN)
else:
    print("MLflow comparison disabled.")

if torch.cuda.is_available():
    torch.cuda.synchronize()
